# Descarga de datasets de FMP API
### Mg. Ing. Diego Martín Méndez


### Importo las librerías

In [30]:
# requests: para realizar solicitudes HTTP a APIs externas
# time: control de pausas/temporizadores (útil en llamadas a APIs)
# dotenv (load_dotenv): carga variables de entorno desde un archivo .env
# os: acceso a variables de entorno y operaciones del sistema
# datetime: manejo de fechas y marcas de tiempo
# pathlib (Path): manejo de rutas de archivos de forma multiplataforma

import requests
import pandas as pd
import numpy as np
import time
from dotenv import load_dotenv
import os
from datetime import datetime
from pathlib import Path

In [31]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [33]:
# Clase de configuración FMPConfig

# Encapsula la lógica de acceso a credenciales sensibles (API Key) necesarias para conectarse a la API de FMP (Financial Modeling Prep), para no exponerlas.

class FMPConfig: 
    def __init__(self, env_file: str = ".env"):
        load_dotenv(dotenv_path=env_file)
        self.api_key = os.getenv("API_KEY")

    def get_api_key(self):
        if not self.api_key:
            raise ValueError("No se encontró 'API_KEY' en el archivo .env ni en las variables de entorno.")
        return self.api_key

In [34]:
# Clases de acceso a la API de Financial Modeling Prep (FMP)

# FMPEndpointStable: apunta a la versión "stable" de la API (https://financialmodelingprep.com/stable/), utilizada para los endpoints más recientes y recomendados por la API.
# FMPEndpointV3: apunta a la versión "v3" de la API (https://financialmodelingprep.com/api/v3/), mantenida por compatibilidad con endpoints más antiguos.

class FMPEndpointStable:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://financialmodelingprep.com/stable/"

    def get(self, endpoint: str, params: dict = None):
        if params is None:
            params = {}
        params["apikey"] = self.api_key
        url = self.base_url + endpoint
        response = requests.get(url, params=params)

        if response.status_code != 200:
            raise Exception(f"Error {response.status_code}: {response.text}")

        return response.json()


class FMPEndpointV3:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://financialmodelingprep.com/api/v3/"

    def get(self, endpoint: str, params: dict = None):
        if params is None:
            params = {}
        params["apikey"] = self.api_key
        url = self.base_url + endpoint
        response = requests.get(url, params=params)

        if response.status_code != 200:
            raise Exception(f"Error {response.status_code}: {response.text}")

        return response.json()

In [41]:
# Genera un archivo parquet con el listado de símbolos bursátiles pertenecientes a las bolsas NYSE y NASDAQ, obtenidos desde FMP API (endpoint "stock/list").

# Instancia FMPConfig para cargar la API Key desde el .env, y FMPEndpointV3 como cliente para consumir la API.
# Descarga el listado completo de símbolos disponibles y lo convierte en un DataFrame de pandas.
# Filtra el resultado, conservando únicamente los registros cuyo "exchangeShortName" sea NYSE o NASDAQ.
# Reduce el DataFrame a las columnas relevantes: "symbol" y "name".

# Parámetro:
# out_path (str): ruta del archivo parquet de salida (por defecto "symbols_nyse_nasdaq.parquet").

# Retorna: 
# Path: la ruta (Path) del archivo parquet generado.

def generar_stock_list_nyse_nasdaq(out_path: str = "symbols_nyse_nasdaq.parquet") -> Path:
    
    # Configuración y Cliente:
    config = FMPConfig()
    api_key = config.get_api_key()

    client = FMPEndpointV3(api_key)

    # Descarga:
    symbols_all = pd.DataFrame(client.get("stock/list"))

    # Filtro por NYSE y NASDAQ:
    symbols_all = symbols_all[symbols_all["exchangeShortName"].isin(["NYSE", "NASDAQ"])].copy()

    # Solo columnas relevantes:
    symbols_nyse_nasdaq = symbols_all[["symbol", "name"]].copy()

    # Guardar en parquet:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    symbols_nyse_nasdaq.to_parquet(out_path, index=False)

    return out_path

In [47]:
generar_stock_list_nyse_nasdaq("archivos/symbols_nyse_nasdaq.parquet")

In [48]:
symbols_nyse_nasdaq = pd.read_parquet("archivos/symbols_nyse_nasdaq.parquet")
symbols_nyse_nasdaq

,symbol,name
0,HMY,Harmony Gold Mining Company Limited
1,MANE,Veradermics Inc.
2,BOND,PIMCO Active Bond Exchange-Traded Fund
3,WTM,"White Mountains Insurance Group, Ltd."
4,SMR,NuScale Power Corporation
...,...,...
18288,UNVR,Univar Solutions Inc.
18289,GOGN-WT,GoGreen Investments Corporation
18290,VHNAW,Vahanna Tech Edge Acquisition I Corp.
18291,TENKR,TenX Keane Acquisition Right


In [6]:
# Generar_profiles_nyse_nasdaq

# Descarga los perfiles ("profile") de cada símbolo bursátil usando los symbol descargados con el endpoint "stock/list".

# Lee el parquet de símbolos indicado en 'symbols_parquet', extrae la columna "symbol", elimina valores nulos, convierte todo a string y descarta duplicados.
# Itera sobre cada símbolo y descarga su perfil.
# Consolida todos los perfiles descargados en un DataFrame, crea el directorio de destino si no existe, y guarda el resultado en formato parquet.

# Parámetros: 
# symbols_parquet (str): ruta al parquet de entrada con la columna "symbol" (por defecto "symbols_nyse_nasdaq.parquet").
# out_path (str): ruta del archivo parquet de salida (por defecto "profiles_nyse_nasdaq.parquet").

# Retorna:
# Path: la ruta (Path) del archivo parquet generado.

def generar_profiles_nyse_nasdaq(symbols_parquet: str = "symbols_nyse_nasdaq.parquet", out_path: str = "profiles_nyse_nasdaq.parquet") -> Path:

    # Cargar símbolos:
    symbols = (pd.read_parquet(symbols_parquet)["symbol"].dropna().astype(str).unique().tolist())

    # Configuración y Cliente:
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)
    
    # Descargar profiles:
    rows = []
    for sym in symbols:
        try:
            data = client.get("profile", params={"symbol": sym})
            if isinstance(data, list):
                rows.extend(data)
            elif isinstance(data, dict):
                rows.append(data)
        except Exception as e:
            pass

    # Guardar parquet:
    profiles_df = pd.DataFrame(rows)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    profiles_df.to_parquet(out_path, index=False)

    return out_path

In [ ]:
generar_profiles_nyse_nasdaq(symbols_parquet="archivos/symbols_nyse_nasdaq.parquet", out_path="archivos/profiles_nyse_nasdaq.parquet")

In [8]:
profiles_nyse_nasdaq = pd.read_parquet("archivos/profiles_nyse_nasdaq.parquet")
profiles_nyse_nasdaq

,symbol,price,marketCap,beta,lastDividend,range,change,changePercentage,volume,averageVolume,...,city,state,zip,image,ipoDate,defaultImage,isEtf,isActivelyTrading,isAdr,isFund
0,NERV,4.7300,3.307674e+07,-0.232000,0.00,1.15-12.46,-0.20000,-4.05680,53639,69775.0,...,Burlington,MA,02451,https://images.financialmodelingprep.com/symbo...,2014-07-01,False,False,True,False,False
1,BIO,305.8850,8.251159e+09,1.180000,0.00,211.43-351.02,6.90501,2.30952,202917,187490.0,...,Hercules,CA,94547,https://images.financialmodelingprep.com/symbo...,1980-02-27,False,False,True,False,False
2,CSCO,81.7150,3.228637e+11,0.865000,1.64,52.11-84.24,-1.39500,-1.67850,31426085,21266630.0,...,San Jose,CA,95134-1706,https://images.financialmodelingprep.com/symbo...,1990-02-16,False,False,True,False,False
3,AKAM,92.1800,1.326150e+10,0.723000,0.00,67.51-104.98,0.39000,0.42488,2317100,3461650.0,...,Cambridge,MA,02142,https://images.financialmodelingprep.com/symbo...,1999-10-29,False,False,True,False,False
4,LUV,52.6900,2.724890e+10,1.107000,0.72,23.82-52.765,1.50000,2.93026,7912532,9632943.0,...,Dallas,TX,75235-1611,https://images.financialmodelingprep.com/symbo...,1980-01-02,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18284,UNVR,36.1400,5.700904e+09,1.668032,0.00,21.49-36.15,0.17000,0.47260,5259018,1977190.0,...,Downers Grove,IL,60515,https://images.financialmodelingprep.com/symbo...,2015-06-17,False,False,False,False,False
18285,GOGN-WT,1.0500,0.000000e+00,0.000000,0.00,1.05-1.19,0.05000,5.00000,16262,0.0,...,None,None,None,https://images.financialmodelingprep.com/symbo...,,True,False,False,False,False
18286,VHNAW,0.1229,0.000000e+00,0.000000,0.00,0.1102-0.17,0.01290,11.72730,711118,0.0,...,New York City,NY,10020,https://images.financialmodelingprep.com/symbo...,,True,False,False,False,False
18287,TENKR,0.4800,3.199680e+06,0.000000,0.00,0.48-0.5,-0.02910,-5.71600,11242,0.0,...,None,None,None,https://images.financialmodelingprep.com/symbo...,,False,False,False,False,False


In [18]:
# analyst_estimates_quarter_nyse_nasdaq

# Descarga las estimaciones de analistas ("analyst-estimates") con periodicidad trimestral ("quarter") para cada symbol en el archivo parquet de entrada, consultando la API de FMP, y guarda el resultado en un nuevo archivo parquet.

# Lee el parquet de símbolos indicado en 'symbols_parquet', extrae la columna "symbol", elimina valores nulos, convierte todo a string y descarta duplicados.
# Instancia FMPConfig para obtener la API Key desde el .env, y FMPEndpointStable como cliente para consumir el endpoint "analyst-estimates" de la API (versión "stable").
# Para cada símbolo, recorre los resultados y solicita la página actual (parámetro "page", comenzando en 0) con un límite de hasta 1000 registros por página ("limit").
# Consolida todas las filas obtenidas en un DataFrame, crea el directorio de destino si no existe, y guarda el resultado en formato parquet.

# Parámetros:
# symbols_parquet (str): ruta al parquet de entrada con la columna "symbol" (por defecto "symbols_nyse_nasdaq.parquet").
# out_path (str): ruta del archivo parquet de salida (por defecto "analyst_estimates_quarter_nyse_nasdaq.parquet").

# Retorna:
# Path: la ruta (Path) del archivo parquet generado.

def analyst_estimates_quarter_nyse_nasdaq(symbols_parquet: str = "symbols_nyse_nasdaq.parquet", out_path: str = "analyst_estimates_quarter_nyse_nasdaq.parquet") -> Path:

    # Leer símbolos:
    symbols = (pd.read_parquet(symbols_parquet)["symbol"].dropna().astype(str).unique().tolist())

    # Cliente stable:
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # Descargar estimates:
    rows = []
    for symbol in symbols:
        page = 0
        while True:
            data = client.get(
                "analyst-estimates",
                params={"symbol": symbol, "period": "quarter", "page": page, "limit": 1000},
            )

            if not data:
                break

            for d in data:
                d["symbol"] = symbol
                rows.append(d)

            page += 1

    # Guardar parquet:
    df_total = pd.DataFrame(rows)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_total.to_parquet(out_path, index=False)

    return out_path

In [19]:
analyst_estimates_quarter_nyse_nasdaq("archivos/symbols_nyse_nasdaq.parquet", "archivos/analyst_estimates_quarter_nyse_nasdaq.parquet")

In [20]:
analyst_estimates_nyse_nasdaq = pd.read_parquet("archivos/analyst_estimates_quarter_nyse_nasdaq.parquet")
analyst_estimates_nyse_nasdaq

,symbol,date,revenueLow,revenueHigh,revenueAvg,ebitdaLow,ebitdaHigh,ebitdaAvg,ebitLow,ebitHigh,...,netIncomeHigh,netIncomeAvg,sgaExpenseLow,sgaExpenseHigh,sgaExpenseAvg,epsAvg,epsHigh,epsLow,numAnalystsRevenue,numAnalystsEps
0,XOM,2030-06-30,83824860668,93355147556,8.872100e+10,1.836839e+10,2.045675e+10,1.944127e+10,1.257114e+10,1.400038e+10,...,1.113340e+10,1.042548e+10,2.703146e+09,3.010475e+09,2.861035e+09,2.46000,2.62704,2.28352,2.0,8
1,XOM,2030-03-31,83896666547,93435117250,8.879700e+10,1.838413e+10,2.047427e+10,1.945793e+10,1.258190e+10,1.401238e+10,...,1.122392e+10,1.051024e+10,2.705462e+09,3.013054e+09,2.863486e+09,2.48000,2.64840,2.30208,2.0,6
2,XOM,2029-12-31,82819578359,92235571841,8.765700e+10,1.814811e+10,2.021141e+10,1.920812e+10,1.242037e+10,1.383248e+10,...,9.458877e+09,8.857420e+09,2.670729e+09,2.974371e+09,2.826724e+09,2.09000,2.23192,1.94006,2.0,4
3,XOM,2029-09-30,82976417516,92410242488,8.782300e+10,1.818247e+10,2.024969e+10,1.924450e+10,1.244390e+10,1.385868e+10,...,1.045455e+10,9.789780e+09,2.675786e+09,2.980004e+09,2.832077e+09,2.31000,2.46686,2.14428,2.0,4
4,XOM,2029-06-30,83014210084,92452331801,8.786300e+10,1.819075e+10,2.025891e+10,1.925326e+10,1.244956e+10,1.386499e+10,...,1.045455e+10,9.789780e+09,2.677005e+09,2.981361e+09,2.833367e+09,2.31000,2.46686,2.14428,2.0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
308142,CSTA,2024-06-30,0,0,0.000000e+00,NaN,NaN,NaN,NaN,NaN,...,1.643846e+07,1.643846e+07,NaN,NaN,NaN,1.34259,1.34259,1.34259,0.0,0
308143,CSTA,2024-03-30,0,0,0.000000e+00,NaN,NaN,NaN,NaN,NaN,...,1.930181e+07,1.930181e+07,NaN,NaN,NaN,1.57645,1.57645,1.57645,0.0,0
308144,CSTA,2023-12-30,0,0,0.000000e+00,NaN,NaN,NaN,NaN,NaN,...,2.557017e+07,2.557017e+07,NaN,NaN,NaN,2.08841,2.08841,2.08841,0.0,0
308145,CSTA,2023-09-30,0,0,0.000000e+00,NaN,NaN,NaN,NaN,NaN,...,1.699592e+07,1.699592e+07,NaN,NaN,NaN,1.38812,1.38812,1.38812,0.0,0


In [21]:
analyst_estimates_nyse_nasdaq["symbol"].nunique()

6962

In [22]:
# generar_income_statements_quarter_nyse_nasdaq

# Descarga los estados de resultados ("income-statement") con periodicidad trimestral ("quarter") para cada symbol en un archivo .parquet de entrada, consultando la API FMP, y guarda el resultado consolidado en un nuevo archivo .parquet.

# Lee el parquet de símbolos indicado en 'symbols_parquet', extrae la columna "symbol", elimina valores nulos, convierte todo a string, lo normaliza a mayúsculas y descarta duplicados.
# Instancia FMPConfig para obtener la API Key desde el .env, y FMPEndpointStable como cliente para consumir el endpoint "income-statement" de la API (versión "stable").

# Crea el directorio de destino si no existe, y guarda el DataFrame consolidado en formato parquet.

# Parámetros:
# symbols_parquet (str): ruta al parquet de entrada con la columna "symbol" (por defecto "archivos/symbols_nyse_nasdaq.parquet").
# out_path (str): ruta del archivo parquet de salida (por defecto "archivos/income_statements_quarter_nyse_nasdaq.parquet").

# Retorna:
# Path: la ruta (Path) del archivo parquet generado.

def generar_income_statements_quarter_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/income_statements_quarter_nyse_nasdaq.parquet") -> Path:

    # Leer símbolos:
    symbols = pd.read_parquet(symbols_parquet)["symbol"].dropna().astype(str).str.upper().unique().tolist()

    # Cliente:
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # Descarga:
    df_total = pd.DataFrame()
    for i, sym in enumerate(symbols, start=1):
        data = client.get("income-statement", params={"symbol": sym, "period": "quarter", "limit": 1000})
        if data:
            df_sym = pd.DataFrame(data)
            df_sym["symbol"] = sym
            df_total = pd.concat([df_total, df_sym], ignore_index=True)

    # Guardar:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_total.to_parquet(out_path, index=False)

    return out_path

In [23]:
generar_income_statements_quarter_nyse_nasdaq()

C:\Users\mging\AppData\Local\Temp\ipykernel_20640\3230753784.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat([df_total, df_sym], ignore_index=True)


In [38]:
income_statements_quarter_nyse_nasdaq = pd.read_parquet("archivos/income_statements_quarter_nyse_nasdaq.parquet")
income_statements_quarter_nyse_nasdaq

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,revenue,costOfRevenue,...,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,netIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil
0,2025-09-30,BTGO,USD,0000000000,2025-09-30,2025-09-30 00:00:00,2025,Q3,5.810456e+09,5.760237e+09,...,2.267200e+07,0.0,-15835000.0,6.837000e+06,0.0,6.837000e+06,0.0000,0.0000,0.000000e+00,0.000000e+00
1,2024-12-31,BTGO,USD,0000000000,2024-12-31,2024-12-31 00:00:00,2024,Q4,1.140324e+09,1.120136e+09,...,1.294010e+08,0.0,-80411000.0,4.899000e+07,0.0,4.899000e+07,0.0000,0.0000,0.000000e+00,0.000000e+00
2,2024-09-30,BTGO,USD,0000000000,2024-09-30,2024-09-30 00:00:00,2024,Q3,8.178260e+08,8.060250e+08,...,-3.752000e+06,0.0,471000.0,-3.281000e+06,0.0,-3.281000e+06,0.0000,0.0000,0.000000e+00,0.000000e+00
3,2025-12-31,XOM,USD,0000034088,2026-01-30,2026-01-30 06:31:24,2025,Q4,8.003900e+10,6.492300e+10,...,6.609000e+09,0.0,0.0,6.501000e+09,0.0,6.501000e+09,1.5000,1.5300,4.331000e+09,4.238000e+09
4,2025-09-30,XOM,USD,0000034088,2025-11-03,2025-11-03 12:45:47,2025,Q3,8.333100e+10,6.464600e+10,...,7.768000e+09,0.0,0.0,7.548000e+09,0.0,7.548000e+09,1.7600,1.7600,4.331000e+09,4.331000e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639941,2021-06-30,VHNAW,USD,0001868640,2021-06-30,2021-06-28 20:00:00,2022,Q1,0.000000e+00,0.000000e+00,...,-1.065600e+04,0.0,0.0,-1.065600e+04,0.0,-1.065600e+04,-0.0004,-0.0004,2.501250e+07,2.501250e+07
639942,2023-03-31,AFTR-WT,USD,0001865975,2023-05-04,2023-05-04 16:31:39,2023,Q1,0.000000e+00,0.000000e+00,...,0.000000e+00,0.0,0.0,2.303400e+05,0.0,2.303400e+05,-0.4000,-0.4000,6.250000e+06,6.250000e+06
639943,2022-09-30,AFTR-WT,USD,0001865975,2022-11-01,2022-11-01 16:30:57,2022,Q3,0.000000e+00,0.000000e+00,...,0.000000e+00,0.0,0.0,2.144000e+06,0.0,2.144000e+06,0.3200,0.3200,1.250000e+07,1.250000e+07
639944,2022-03-31,AFTR-WT,USD,0001865975,2022-05-05,2022-05-05 16:51:30,2022,Q1,0.000000e+00,0.000000e+00,...,0.000000e+00,0.0,0.0,2.577000e+06,0.0,2.577000e+06,0.1000,0.1000,2.500000e+07,2.500000e+07


In [39]:
income_statements_quarter_nyse_nasdaq.columns

Index(['date', 'symbol', 'reportedCurrency', 'cik', 'filingDate',
       'acceptedDate', 'fiscalYear', 'period', 'revenue', 'costOfRevenue',
       'grossProfit', 'researchAndDevelopmentExpenses',
       'generalAndAdministrativeExpenses', 'sellingAndMarketingExpenses',
       'sellingGeneralAndAdministrativeExpenses', 'otherExpenses',
       'operatingExpenses', 'costAndExpenses', 'netInterestIncome',
       'interestIncome', 'interestExpense', 'depreciationAndAmortization',
       'ebitda', 'ebit', 'nonOperatingIncomeExcludingInterest',
       'operatingIncome', 'totalOtherIncomeExpensesNet', 'incomeBeforeTax',
       'incomeTaxExpense', 'netIncomeFromContinuingOperations',
       'netIncomeFromDiscontinuedOperations', 'otherAdjustmentsToNetIncome',
       'netIncome', 'netIncomeDeductions', 'bottomLineNetIncome', 'eps',
       'epsDiluted', 'weightedAverageShsOut', 'weightedAverageShsOutDil'],
      dtype='object')

In [114]:
income_statements_quarter_nyse_nasdaq["symbol"].nunique()

11808

In [49]:
# generar_balance_sheet_statements_quarter_nyse_nasdaq

# Descarga los balances generales ("balance-sheet-statement") con periodicidad trimestral ("quarter") para cada symbol en un archivo .parquet de entrada, consultando la API de FMP, y guarda el resultado consolidado en un nuevo archivo .parquet.

# Lee el parquet de símbolos indicado en 'symbols_parquet', extrae la columna "symbol", elimina valores nulos, convierte todo a string, lo normaliza a mayúsculas y descarta duplicados.
# Instancia FMPConfig para obtener la API Key desde el .env, y FMPEndpointStable como cliente para consumir el endpoint "balance-sheet-statement" de la API (versión "stable").
# Guarda el DataFrame consolidado en formato parquet.

# Parámetros:
# symbols_parquet (str): ruta al parquet de entrada con la columna "symbol" (por defecto "archivos/symbols_nyse_nasdaq.parquet").
# out_path (str): ruta del archivo parquet de salida (por defecto "archivos/balance_sheet_statements_quarter_nyse_nasdaq.parquet").

# Retorna:
# Path: la ruta (Path) del archivo parquet generado.

def generar_balance_sheet_statements_quarter_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/balance_sheet_statements_quarter_nyse_nasdaq.parquet") -> Path:

    # Leer símbolos:
    symbols = pd.read_parquet(symbols_parquet)["symbol"].dropna().astype(str).str.upper().unique().tolist()

    # Cliente:
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # Descarga:
    df_total = pd.DataFrame()
    for i, sym in enumerate(symbols, start=1):
        data = client.get("balance-sheet-statement", params={"symbol": sym, "period": "quarter", "limit": 1000})
        if data:
            df_sym = pd.DataFrame(data)
            df_sym["symbol"] = sym
            df_total = pd.concat([df_total, df_sym], ignore_index=True)

    # Guardar:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_total.to_parquet(out_path, index=False)

    return out_path

In [50]:
generar_balance_sheet_statements_quarter_nyse_nasdaq()

C:\Users\mging\AppData\Local\Temp\ipykernel_20640\1851097330.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat([df_total, df_sym], ignore_index=True)


In [55]:
balance_sheet_statements_quarter_nyse_nasdaq = pd.read_parquet("archivos/balance_sheet_statements_quarter_nyse_nasdaq.parquet")
balance_sheet_statements_quarter_nyse_nasdaq

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,cashAndCashEquivalents,shortTermInvestments,...,additionalPaidInCapital,accumulatedOtherComprehensiveIncomeLoss,otherTotalStockholdersEquity,totalStockholdersEquity,totalEquity,minorityInterest,totalLiabilitiesAndTotalEquity,totalInvestments,totalDebt,netDebt
0,2025-06-30,HMY,ZAR,0001023514,2025-08-28,2025-08-28 10:30:09,2025,Q4,1.310100e+10,0.0,...,0.0,7.170000e+08,0.0,4.823500e+10,4.851200e+10,277000000.0,7.750300e+10,1.970000e+08,2.229000e+09,-1.087200e+10
1,2024-12-31,HMY,ZAR,0001023514,2024-12-31,2024-12-31 00:00:00,2025,Q2,9.396000e+09,0.0,...,0.0,3.394000e+09,0.0,4.582600e+10,4.604400e+10,218000000.0,6.889600e+10,1.400000e+08,2.027000e+09,-7.369000e+09
2,2024-06-30,HMY,ZAR,0001023514,2024-10-31,2024-10-31 12:23:52,2024,Q4,4.693000e+09,39000000.0,...,0.0,6.081000e+09,-479000000.0,4.077400e+10,4.094900e+10,175000000.0,6.046000e+10,2.920000e+08,2.291000e+09,-2.402000e+09
3,2023-12-31,HMY,ZAR,0001023514,2023-12-31,2023-12-29 19:00:00,2024,Q2,3.477000e+09,241000000.0,...,0.0,6.399000e+09,0.0,3.983300e+10,3.997500e+10,142000000.0,5.978600e+10,6.840000e+09,3.362000e+09,-1.150000e+08
4,2023-06-30,HMY,ZAR,0001023514,2023-06-30,2023-06-29 20:00:00,2023,Q4,2.867000e+09,0.0,...,0.0,6.778000e+09,0.0,3.475700e+10,3.488000e+10,123000000.0,5.724000e+10,1.110000e+08,5.695000e+09,2.828000e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
614323,2021-03-31,TENKR,USD,0001851484,2021-03-31,2021-03-31 00:00:00,2021,Q1,0.000000e+00,0.0,...,0.0,0.000000e+00,-144.0,-9.389000e+03,-9.389000e+03,0.0,4.400000e+04,0.000000e+00,0.000000e+00,0.000000e+00
614324,2023-03-31,AFTR-WT,USD,0001865975,2023-05-04,2023-05-04 16:31:39,2023,Q1,3.390580e+05,0.0,...,0.0,0.000000e+00,0.0,-1.251600e+07,0.000000e+00,0.0,5.985310e+05,2.563470e+08,0.000000e+00,-3.390000e+05
614325,2022-09-30,AFTR-WT,USD,0001865975,2022-11-01,2022-11-01 16:30:57,2022,Q3,9.370750e+05,0.0,...,0.0,0.000000e+00,0.0,-1.162700e+07,0.000000e+00,0.0,1.414000e+06,2.514770e+08,0.000000e+00,-9.370000e+05
614326,2022-03-31,AFTR-WT,USD,0001865975,2022-05-05,2022-05-05 16:51:30,2022,Q1,1.523000e+06,0.0,...,0.0,0.000000e+00,0.0,-1.432600e+07,0.000000e+00,0.0,2.139000e+06,0.000000e+00,0.000000e+00,-1.523000e+06


In [56]:
balance_sheet_statements_quarter_nyse_nasdaq["symbol"].nunique()

12415

In [57]:
def generar_cash_flow_statements_quarter_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/cash_flow_statements_quarter_nyse_nasdaq.parquet") -> Path:
        
    """
    Descarga todos los income statements trimestrales para los símbolos del parquet
    y guarda un único DataFrame consolidado en parquet.
    """
    # Leer symbols:
    symbols = pd.read_parquet(symbols_parquet)["symbol"].dropna().astype(str).str.upper().unique().tolist()

    # Cliente:
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # Descarga:
    df_total = pd.DataFrame()
    for i, sym in enumerate(symbols, start=1):
        data = client.get("cash-flow-statement", params={"symbol": sym, "period": "quarter", "limit": 1000})
        if data:
            df_sym = pd.DataFrame(data)
            df_sym["symbol"] = sym
            df_total = pd.concat([df_total, df_sym], ignore_index=True)

    # Guardar
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_total.to_parquet(out_path, index=False)

    return out_path

In [58]:
generar_cash_flow_statements_quarter_nyse_nasdaq()

C:\Users\mging\AppData\Local\Temp\ipykernel_20640\1874836709.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat([df_total, df_sym], ignore_index=True)


In [59]:
cash_flow_statements_quarter_nyse_nasdaq = pd.read_parquet("archivos/cash_flow_statements_quarter_nyse_nasdaq.parquet")
cash_flow_statements_quarter_nyse_nasdaq

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,netIncome,depreciationAndAmortization,...,netCashProvidedByFinancingActivities,effectOfForexChangesOnCash,netChangeInCash,cashAtEndOfPeriod,cashAtBeginningOfPeriod,operatingCashFlow,capitalExpenditure,freeCashFlow,incomeTaxesPaid,interestPaid
0,2025-06-30,HMY,ZAR,0001023514,2025-08-28,2025-08-28 10:30:09,2025,Q4,6.527000e+09,2.414000e+09,...,-1.663000e+09,-109000000.0,-9.396000e+09,0.000000e+00,9.396000e+09,1.246200e+10,-7.049000e+09,5.413000e+09,0.0,124000000.0
1,2024-12-31,HMY,ZAR,0001023514,2024-12-31,2024-12-31 00:00:00,2025,Q2,7.857000e+09,2.428000e+09,...,-5.520000e+08,40000000.0,9.396000e+09,9.396000e+09,0.000000e+00,1.018500e+10,-4.806000e+09,5.379000e+09,0.0,134000000.0
2,2024-06-30,HMY,ZAR,0001023514,2024-10-31,2024-10-31 12:23:52,2024,Q4,2.667000e+09,2.211000e+09,...,-2.691000e+09,-74000000.0,1.255000e+09,4.732000e+09,3.477000e+09,8.655000e+09,-4.530000e+09,4.125000e+09,0.0,0.0
3,2023-12-31,HMY,ZAR,0001023514,2023-12-31,2023-12-29 19:00:00,2024,Q2,5.920000e+09,2.381000e+09,...,-2.744000e+09,46000000.0,5.690000e+08,3.477000e+09,2.908000e+09,6.995000e+09,-3.868000e+09,3.127000e+09,0.0,0.0
4,2023-06-30,HMY,ZAR,0001023514,2023-06-30,2023-06-29 20:00:00,2023,Q4,2.981000e+09,1.665000e+09,...,-2.131000e+09,-4330154.0,6.790000e+08,2.908000e+09,2.229000e+09,6.883000e+09,-3.994000e+09,2.889000e+09,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
605185,2023-12-31,TENKR,USD,0001851484,2024-04-16,2024-04-16 17:00:38,2023,Q4,1.033339e+06,0.000000e+00,...,8.750840e+05,2703.0,6.222000e+03,3.274500e+04,2.652300e+04,-2.181900e+05,0.000000e+00,-2.181900e+05,0.0,0.0
605186,2023-09-30,TENKR,USD,0001851484,2023-11-20,2023-11-20 17:25:25,2023,Q3,1.579430e+05,0.000000e+00,...,4.299000e+05,-19598.0,-3.435720e+05,2.652300e+04,3.700950e+05,-1.134720e+05,0.000000e+00,-1.134720e+05,0.0,0.0
605187,2023-06-30,TENKR,USD,0001851484,2023-08-16,2023-08-16 14:44:39,2023,Q2,6.162960e+05,0.000000e+00,...,3.455360e+05,0.0,2.497190e+05,3.700950e+05,1.203760e+05,-1.002560e+05,0.000000e+00,-1.002560e+05,0.0,0.0
605188,2023-03-31,TENKR,USD,0001851484,2023-05-15,2023-05-15 16:19:32,2023,Q1,6.117250e+05,0.000000e+00,...,0.000000e+00,0.0,-1.687990e+05,1.203760e+05,2.891750e+05,-1.687990e+05,0.000000e+00,-1.687990e+05,0.0,0.0


In [116]:
cash_flow_statements_quarter_nyse_nasdaq["symbol"].nunique()

11773

In [5]:
def generar_key_metrics_quarter_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/key_metrics_quarter_nyse_nasdaq.parquet") -> Path:
    """
    Descarga todos los key metrics trimestrales para los símbolos del parquet
    y guarda un único DataFrame consolidado en parquet.
    """
    # Leer símbolos:
    symbols = (
        pd.read_parquet(symbols_parquet)["symbol"]
        .dropna().astype(str).str.upper().unique().tolist()
    )

    # Cliente:
    client = FMPEndpointStable(FMPConfig().get_api_key())

    # Descarga con manejo de timeout:
    frames = []
    for sym in symbols:
        try:
            data = client.get(
                "key-metrics",
                params={"symbol": sym, "period": "quarter", "limit": 1000, "timeout": 30}
            )
            if data:
                df_sym = pd.DataFrame(data)
                df_sym["symbol"] = sym
                frames.append(df_sym)
        except requests.exceptions.ReadTimeout:
            continue
        except Exception as e:
            continue

        # opcional: dormir un poco entre requests para aliviar API
        time.sleep(0.2)

    df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # Guardar con seguridad:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    string_cols_keep = {"symbol", "date", "period", "reportedCurrency"}
    INT64_MAX = 2**63 - 1

    for col in df_total.columns:
        if df_total[col].dtype == "object" and col not in string_cols_keep:
            try:
                series = pd.to_numeric(df_total[col], errors="coerce")
                if series.max(skipna=True) > INT64_MAX:
                    df_total[col] = df_total[col].astype(str)
                else:
                    df_total[col] = series
            except Exception:
                df_total[col] = df_total[col].astype(str)

    df_total.to_parquet(out_path, index=False, engine="pyarrow")

    return out_path

In [6]:
generar_key_metrics_quarter_nyse_nasdaq()

C:\Users\mging\AppData\Local\Temp\ipykernel_15488\1905155840.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [7]:
key_metrics_quarter_nyse_nasdaq = pd.read_parquet("archivos/key_metrics_quarter_nyse_nasdaq.parquet")
key_metrics_quarter_nyse_nasdaq

,symbol,date,fiscalYear,period,reportedCurrency,marketCap,enterpriseValue,evToSales,evToOperatingCashFlow,evToFreeCashFlow,...,averageInventory,daysOfSalesOutstanding,daysOfPayablesOutstanding,daysOfInventoryOutstanding,operatingCycle,cashConversionCycle,freeCashFlowToEquity,freeCashFlowToFirm,tangibleAssetValue,netCurrentAssetValue
0,HMY,2025-06-30,2025,Q4,ZAR,156100649456.32358,145228649456.32358,3.951262,11.653719,26.829605,...,3.673000e+09,9.799483,27.467320,15.625000,25.424483,-2.042837,1.628500e+10,1.078405e+09,4.850600e+10,-7.685000e+09
1,HMY,2024-12-31,2025,Q2,ZAR,96903156754.92593,89534156754.92593,2.410656,8.790786,16.645130,...,3.562000e+09,9.457742,22.706440,14.045920,23.503662,0.797222,1.274800e+10,-1.160109e+09,4.603200e+10,-5.734000e+09
2,HMY,2024-06-30,2024,Q4,ZAR,104929788361.0487,102527788361.0487,3.438799,11.846076,24.855221,...,3.408000e+09,6.788865,22.933907,14.679493,21.468358,-1.465549,6.527000e+09,3.311514e+09,4.093000e+10,-8.014000e+09
3,HMY,2023-12-31,2024,Q2,ZAR,69549626760.0,69434626760.0,2.199804,9.926323,22.204869,...,3.239000e+09,9.372386,19.925786,12.404873,21.777259,1.851473,3.242000e+09,2.295419e+09,3.994900e+10,-9.593000e+09
4,HMY,2023-06-30,2023,Q4,ZAR,0.0,2828000000.0,0.000000,0.410867,0.978885,...,3.004500e+09,0.000000,0.000000,0.000000,0.000000,0.000000,6.100000e+07,0.000000e+00,3.484700e+10,-1.368200e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
651074,TENKR,2021-03-31,2021,Q1,USD,0.0,0.0,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,-9.389000e+03,-5.338900e+04
651075,AFTR-WT,2023-03-31,2023,Q1,USD,63875000.00000001,63535942.00000001,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,2.438320e+08,-1.251547e+07
651076,AFTR-WT,2022-09-30,2022,Q3,USD,122125000.0,121187925.0,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,2.398500e+08,-1.162700e+07
651077,AFTR-WT,2022-03-31,2022,Q1,USD,242000000.0,240477000.0,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,2.356740e+08,-1.432600e+07


In [8]:
key_metrics_quarter_nyse_nasdaq["symbol"].nunique()

12438

In [9]:
def generar_ratios_quarter_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/ratios_quarter_nyse_nasdaq.parquet") -> Path:
    """
    Descarga todos los ratios trimestrales para los símbolos del parquet
    y guarda un único DataFrame consolidado en parquet.
    """
    
    # Leer símbolos:
    symbols = (
        pd.read_parquet(symbols_parquet)["symbol"]
        .dropna().astype(str).str.upper().unique().tolist()
    )

    # Cliente:
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # Descarga con manejo de timeout:
    frames = []
    for sym in symbols:
        try:
            data = client.get(
                "ratios",
                params={"symbol": sym, "period": "quarter", "limit": 1000, "timeout": 30}
            )
            if data:
                df_sym = pd.DataFrame(data)
                df_sym["symbol"] = sym
                frames.append(df_sym)
        except requests.exceptions.ReadTimeout:
            continue
        except Exception as e:
            continue

        # opcional: dormir un poco entre requests para aliviar API
        time.sleep(0.2)

    df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # Guardar con seguridad:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    string_cols_keep = {"symbol", "date", "period", "reportedCurrency"}
    INT64_MAX = 2**63 - 1

    for col in df_total.columns:
        if df_total[col].dtype == "object" and col not in string_cols_keep:
            try:
                series = pd.to_numeric(df_total[col], errors="coerce")
                if series.max(skipna=True) > INT64_MAX:
                    df_total[col] = df_total[col].astype(str)
                else:
                    df_total[col] = series
            except Exception:
                df_total[col] = df_total[col].astype(str)

    df_total.to_parquet(out_path, index=False, engine="pyarrow")

    return out_path

In [10]:
generar_ratios_quarter_nyse_nasdaq()

In [37]:
ratios_quarter_nyse_nasdaq = pd.read_parquet("archivos/ratios_quarter_nyse_nasdaq.parquet")
ratios_quarter_nyse_nasdaq

,symbol,date,fiscalYear,period,reportedCurrency,grossProfitMargin,ebitMargin,ebitdaMargin,operatingProfitMargin,pretaxProfitMargin,...,operatingCashFlowPerShare,capexPerShare,freeCashFlowPerShare,netIncomePerEBT,ebtPerEbit,priceToFairValue,debtToMarketCap,effectiveTaxRate,enterpriseValueMultiple,dividendPerShare
0,HMY,2025-06-30,2025,Q4,ZAR,0.400571,0.304421,0.370099,0.281703,0.295388,...,19.712148,11.149970,8.562178,0.601179,1.048580,3.236253,0.012511,0.390347,10.676222116909768,2.309399
1,HMY,2024-12-31,2025,Q2,ZAR,0.392558,0.275545,0.340917,0.269325,0.278641,...,16.294444,7.688866,8.605578,0.759204,1.034590,2.114589,0.021805,0.233839,7.071091198462007,0.955109
2,HMY,2024-06-30,2024,Q4,ZAR,0.259098,0.153010,0.227168,0.226329,0.142680,...,13.680871,7.160525,6.520346,0.626939,0.630409,2.573448,0.017097,0.358721,15.137721594721498,2.271451
3,HMY,2023-12-31,2024,Q2,ZAR,0.269864,0.231783,0.308896,0.156127,0.239545,...,11.300485,6.248788,5.051696,0.782965,1.534294,1.746030,0.048340,0.205793,7.121500180512821,0.020535
4,HMY,2023-06-30,2023,Q4,ZAR,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
651276,TENKR,2021-03-31,2021,Q1,USD,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
651277,AFTR-WT,2023-03-31,2023,Q1,USD,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,-5.103468,0.000000,0.000000,0,0.000000
651278,AFTR-WT,2022-09-30,2022,Q3,USD,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,-10.503569,0.000000,0.000000,0,0.000000
651279,AFTR-WT,2022-03-31,2022,Q1,USD,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,-16.892364,0.000000,0.000000,0,0.000000


In [38]:
ratios_quarter_nyse_nasdaq["symbol"].nunique()

12440

In [12]:
def generar_enterprise_values_quarter_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/enterprise_values_quarter_nyse_nasdaq.parquet") -> Path:
    
    """
    Descarga todos los ratios trimestrales para los símbolos del parquet
    y guarda un único DataFrame consolidado en parquet.
    """
    
    # Leer símbolos:
    symbols = (
        pd.read_parquet(symbols_parquet)["symbol"]
        .dropna().astype(str).str.upper().unique().tolist()
    )

    # Cliente:
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # Descarga con manejo de timeout:
    frames = []
    for sym in symbols:
        try:
            data = client.get(
                "enterprise-values",
                params={"symbol": sym, "period": "quarter", "limit": 1000, "timeout": 30}
            )
            if data:
                df_sym = pd.DataFrame(data)
                df_sym["symbol"] = sym
                frames.append(df_sym)
        except requests.exceptions.ReadTimeout:
            continue
        except Exception as e:
            continue

        # opcional: dormir un poco entre requests para aliviar API
        time.sleep(0.2)

    df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # Guardar con seguridad:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    string_cols_keep = {"symbol", "date"}
    INT64_MAX = 2**63 - 1

    for col in df_total.columns:
        if df_total[col].dtype == "object" and col not in string_cols_keep:
            try:
                series = pd.to_numeric(df_total[col], errors="coerce")
                if series.max(skipna=True) > INT64_MAX:
                    df_total[col] = df_total[col].astype(str)
                else:
                    df_total[col] = series
            except Exception:
                df_total[col] = df_total[col].astype(str)

    df_total.to_parquet(out_path, index=False, engine="pyarrow")

    return out_path

In [13]:
generar_enterprise_values_quarter_nyse_nasdaq()

C:\Users\mging\AppData\Local\Temp\ipykernel_15488\1316549528.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [14]:
enterprise_values_quarter_nyse_nasdaq = pd.read_parquet("archivos/enterprise_values_quarter_nyse_nasdaq.parquet")
enterprise_values_quarter_nyse_nasdaq

,symbol,date,stockPrice,numberOfShares,marketCapitalization,minusCashAndCashEquivalents,addTotalDebt,enterpriseValue
0,HMY,2025-06-30,246.92,632198987.0,156100649456,1.310100e+10,2.229000e+09,145228649456
1,HMY,2024-12-31,155.03,625059665.0,96903156754,9.396000e+09,2.027000e+09,89534156754
2,HMY,2024-06-30,165.86,632635150.0,104929788361,4.693000e+09,2.291000e+09,102527788361
3,HMY,2023-12-31,112.36,619000000.0,69549626760,3.477000e+09,3.362000e+09,69434626760
4,HMY,2023-06-30,78.99,617000000.0,48738273780,2.867000e+09,5.695000e+09,51566273780
...,...,...,...,...,...,...,...,...
651557,TENKR,2021-03-31,10.09,0.0,0,0.000000e+00,0.000000e+00,0
651558,AFTR-WT,2023-03-31,10.22,6250000.0,63875000,3.390580e+05,0.000000e+00,63535942
651559,AFTR-WT,2022-09-30,9.77,12500000.0,122125000,9.370750e+05,0.000000e+00,121187925
651560,AFTR-WT,2022-03-31,9.68,25000000.0,242000000,1.523000e+06,0.000000e+00,240477000


In [119]:
enterprise_values_quarter_nyse_nasdaq["symbol"].nunique()

11945

In [15]:
def generar_owner_earnings_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/owner_earnings_nyse_nasdaq.parquet") -> Path:
    """
    Descarga todos los income statements trimestrales para los símbolos del parquet
    y guarda un único DataFrame consolidado en parquet.
    """
    # Leer símbolos:
    symbols = pd.read_parquet(symbols_parquet)["symbol"].dropna().astype(str).str.upper().unique().tolist()

    # Cliente:
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # Descarga:
    df_total = pd.DataFrame()
    for i, sym in enumerate(symbols, start=1):
        data = client.get("owner-earnings", params={"symbol": sym , "limit": 1000})
        if data:
            df_sym = pd.DataFrame(data)
            df_sym["symbol"] = sym
            df_total = pd.concat([df_total, df_sym], ignore_index=True)

    # Guardar:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_total.to_parquet(out_path, index=False)

    return out_path

In [16]:
generar_owner_earnings_nyse_nasdaq()

In [17]:
owner_earnings_nyse_nasdaq = pd.read_parquet("archivos/owner_earnings_nyse_nasdaq.parquet")
owner_earnings_nyse_nasdaq

,symbol,reportedCurrency,fiscalYear,period,date,averagePPE,maintenanceCapex,ownersEarnings,growthCapex,ownersEarningsPerShare
0,HMY,ZAR,2025,Q4,2025-06-30,0.785640,3.796760e+09,1.625876e+10,-1.084576e+10,25.69000
1,HMY,ZAR,2025,Q2,2024-12-31,0.785640,7.205000e+09,1.739000e+10,-1.201100e+10,27.82000
2,HMY,ZAR,2024,Q4,2024-06-30,0.872040,3.910921e+09,1.256592e+10,-8.440921e+09,19.86000
3,HMY,ZAR,2024,Q2,2023-12-31,0.872040,4.877457e+09,1.187246e+10,-8.745457e+09,19.12000
4,HMY,ZAR,2023,Q2,2022-12-31,0.984750,5.647428e+09,8.712428e+09,-9.293428e+09,14.05000
...,...,...,...,...,...,...,...,...,...,...
282412,UNVR,USD,2013,Q3,2013-09-30,0.090514,4.646173e+07,1.015617e+08,-7.326173e+07,0.73630
282413,UNVR,USD,2013,Q2,2013-06-30,0.090514,3.304634e+07,1.502463e+08,-8.094634e+07,1.09000
282414,UNVR,USD,2013,Q1,2013-03-31,0.090514,2.756903e+07,1.876903e+07,-6.326903e+07,0.13607
282415,VHNAW,USD,2025,Q4,2025-12-31,0.036217,-7.433174e+11,-7.433084e+11,-8.219023e+10,-9434.40000


In [120]:
owner_earnings_nyse_nasdaq["symbol"].nunique()

8114

In [22]:
def generar_income_statement_growth_quarter_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/income_statement_growth_quarter_nyse_nasdaq.parquet") -> Path:
    """
    Descarga todos los key metrics trimestrales para los símbolos del parquet
    y guarda un único DataFrame consolidado en parquet.
    """
    # Leer símbolos:
    symbols = (
        pd.read_parquet(symbols_parquet)["symbol"]
        .dropna().astype(str).str.upper().unique().tolist()
    )

    # Cliente:
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # Descarga con manejo de timeout:
    frames = []
    for sym in symbols:
        try:
            data = client.get(
                "income-statement-growth",
                params={"symbol": sym, "period": "quarter", "limit": 1000, "timeout": 30}
            )
            if data:
                df_sym = pd.DataFrame(data)
                df_sym["symbol"] = sym
                frames.append(df_sym)
        except requests.exceptions.ReadTimeout:
            continue
        except Exception as e:
            continue

        # opcional: dormir un poco entre requests para aliviar API
        time.sleep(0.2)

    df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # 4) Guardar con seguridad
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    string_cols_keep = {"symbol", "date", "period", "reportedCurrency"}
    INT64_MAX = 2**63 - 1

    for col in df_total.columns:
        if df_total[col].dtype == "object" and col not in string_cols_keep:
            try:
                series = pd.to_numeric(df_total[col], errors="coerce")
                if series.max(skipna=True) > INT64_MAX:
                    df_total[col] = df_total[col].astype(str)
                else:
                    df_total[col] = series
            except Exception:
                df_total[col] = df_total[col].astype(str)

    df_total.to_parquet(out_path, index=False, engine="pyarrow")

    return out_path

In [23]:
generar_income_statement_growth_quarter_nyse_nasdaq()

C:\Users\mging\AppData\Local\Temp\ipykernel_15488\3250816084.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [24]:
income_statement_growth_quarter_nyse_nasdaq = pd.read_parquet("archivos/income_statement_growth_quarter_nyse_nasdaq.parquet")
income_statement_growth_quarter_nyse_nasdaq

,symbol,date,fiscalYear,period,reportedCurrency,growthRevenue,growthCostOfRevenue,growthGrossProfit,growthGrossProfitRatio,growthResearchAndDevelopmentExpenses,...,growthEPSDiluted,growthWeightedAverageShsOut,growthWeightedAverageShsOutDil,growthEBIT,growthNonOperatingIncomeExcludingInterest,growthNetInterestIncome,growthTotalOtherIncomeExpensesNet,growthNetIncomeFromContinuingOperations,growthOtherAdjustmentsToNetIncome,growthNetIncomeDeductions
0,HMY,2025-06-30,2025,Q4,ZAR,-0.010393,-0.023448,0.009808,0.020413,0.0,...,-0.178998,0.011422,0.012800,0.093316,-2.614719,0.526646,0.453757,-0.165216,0.0,0.0
1,HMY,2024-12-31,2025,Q2,ZAR,0.245715,0.021322,0.887379,0.515096,0.0,...,1.978673,-0.011974,-0.012069,1.243314,-1.105672,2.127451,1.138733,1.906525,0.0,0.0
2,HMY,2024-06-30,2024,Q4,ZAR,-0.055411,-0.052379,-0.093097,-0.039896,0.0,...,-0.557188,0.022028,0.018736,-0.376435,1.915410,2.146067,-1.947209,-0.545712,-1.0,0.0
3,HMY,2023-12-31,2024,Q2,ZAR,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
4,HMY,2022-12-31,2023,Q2,ZAR,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
643277,TENKR,2022-06-30,2022,Q2,USD,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
643278,AFTR-WT,2023-03-31,2023,Q1,USD,0.000000,0.000000,0.000000,0.000000,0.0,...,-2.250000,-0.500000,-0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
643279,AFTR-WT,2022-09-30,2022,Q3,USD,0.000000,0.000000,0.000000,0.000000,0.0,...,2.200000,-0.500000,-0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
643280,AFTR-WT,2022-03-31,2022,Q1,USD,0.000000,0.000000,0.000000,0.000000,0.0,...,-0.411765,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0


In [31]:
def generar_balance_sheet_statement_growth_quarter_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/balance_sheet_statement_growth_quarter_nyse_nasdaq.parquet") -> Path:
    """
    Descarga todos los key metrics trimestrales para los símbolos del parquet
    y guarda un único DataFrame consolidado en parquet.
    """
    # 1) Leer símbolos
    symbols = (
        pd.read_parquet(symbols_parquet)["symbol"]
        .dropna().astype(str).str.upper().unique().tolist()
    )

    # 2) Cliente
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # 3) Descarga con manejo de timeout
    frames = []
    for sym in symbols:
        try:
            data = client.get(
                "balance-sheet-statement-growth",
                params={"symbol": sym, "period": "quarter", "limit": 1000, "timeout": 30}
            )
            if data:
                df_sym = pd.DataFrame(data)
                df_sym["symbol"] = sym
                frames.append(df_sym)
        except requests.exceptions.ReadTimeout:
            continue
        except Exception as e:
            continue

        # opcional: dormir un poco entre requests para aliviar API
        time.sleep(0.2)

    df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # 4) Guardar con seguridad
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    string_cols_keep = {"symbol", "date", "period", "reportedCurrency"}
    INT64_MAX = 2**63 - 1

    for col in df_total.columns:
        if df_total[col].dtype == "object" and col not in string_cols_keep:
            try:
                series = pd.to_numeric(df_total[col], errors="coerce")
                if series.max(skipna=True) > INT64_MAX:
                    df_total[col] = df_total[col].astype(str)
                else:
                    df_total[col] = series
            except Exception:
                df_total[col] = df_total[col].astype(str)

    df_total.to_parquet(out_path, index=False, engine="pyarrow")

    return out_path

In [32]:
generar_balance_sheet_statement_growth_quarter_nyse_nasdaq()

C:\Users\mging\AppData\Local\Temp\ipykernel_15488\1106029845.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [33]:
balance_sheet_statement_growth_quarter_nyse_nasdaq = pd.read_parquet("archivos/balance_sheet_statement_growth_quarter_nyse_nasdaq.parquet")
balance_sheet_statement_growth_quarter_nyse_nasdaq

,symbol,date,fiscalYear,period,reportedCurrency,growthCashAndCashEquivalents,growthShortTermInvestments,growthCashAndShortTermInvestments,growthNetReceivables,growthInventory,...,growthNetDebt,growthAccountsReceivables,growthOtherReceivables,growthPrepaids,growthTotalPayables,growthOtherPayables,growthAccruedExpenses,growthCapitalLeaseObligationsCurrent,growthAdditionalPaidInCapital,growthTreasuryStock
0,HMY,2025-06-30,2025,Q4,ZAR,0.394317,0.000000,0.394317,0.025365,0.086339,...,-0.475370,0.025365,0.0,0.000000,0.181307,0.0,0.0,0.0,0.0,0.0
1,HMY,2024-12-31,2025,Q2,ZAR,1.002131,-1.000000,0.985630,0.735438,-0.022759,...,-2.067860,1.733193,-1.0,-1.000000,-0.336597,-1.0,-1.0,-1.0,0.0,0.0
2,HMY,2024-06-30,2024,Q4,ZAR,0.349727,-0.838174,0.272727,-0.315789,0.121382,...,-19.886957,-0.565561,0.0,2.258865,0.662469,0.0,0.0,0.0,0.0,0.0
3,HMY,2023-12-31,2024,Q2,ZAR,0.212766,0.000000,0.296826,0.490027,-0.015926,...,-1.040665,1.301821,-1.0,-2.492063,2.329677,-1.0,-1.0,-1.0,0.0,0.0
4,HMY,2023-06-30,2023,Q4,ZAR,0.306150,0.000000,0.306150,-0.054031,0.189869,...,-0.399575,-0.387650,0.0,0.000000,-0.662089,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617539,TENKR,2021-03-31,2021,Q1,USD,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
617540,AFTR-WT,2023-03-31,2023,Q1,USD,-0.638174,0.000000,-0.638174,0.000000,0.000000,...,0.638207,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
617541,AFTR-WT,2022-09-30,2022,Q3,USD,-0.384718,0.000000,-0.384718,0.000000,0.000000,...,0.384767,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
617542,AFTR-WT,2022-03-31,2022,Q1,USD,-0.219375,0.000000,-0.219375,0.000000,0.000000,...,0.219375,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


In [34]:
def generar_cash_flow_statement_growth_quarter_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/cash_flow_statement_growth_quarter_nyse_nasdaq.parquet") -> Path:
    """
    Descarga todos los key metrics trimestrales para los símbolos del parquet
    y guarda un único DataFrame consolidado en parquet.
    """
    # 1) Leer símbolos
    symbols = (
        pd.read_parquet(symbols_parquet)["symbol"]
        .dropna().astype(str).str.upper().unique().tolist()
    )

    # 2) Cliente
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # 3) Descarga con manejo de timeout
    frames = []
    for sym in symbols:
        try:
            data = client.get(
                "cash-flow-statement-growth",
                params={"symbol": sym, "period": "quarter", "limit": 1000, "timeout": 30}
            )
            if data:
                df_sym = pd.DataFrame(data)
                df_sym["symbol"] = sym
                frames.append(df_sym)
        except requests.exceptions.ReadTimeout:
            continue
        except Exception as e:
            continue

        # opcional: dormir un poco entre requests para aliviar API
        time.sleep(0.2)

    df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # 4) Guardar con seguridad
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    string_cols_keep = {"symbol", "date", "period", "reportedCurrency"}
    INT64_MAX = 2**63 - 1

    for col in df_total.columns:
        if df_total[col].dtype == "object" and col not in string_cols_keep:
            try:
                series = pd.to_numeric(df_total[col], errors="coerce")
                if series.max(skipna=True) > INT64_MAX:
                    df_total[col] = df_total[col].astype(str)
                else:
                    df_total[col] = series
            except Exception:
                df_total[col] = df_total[col].astype(str)

    df_total.to_parquet(out_path, index=False, engine="pyarrow")

    return out_path

In [35]:
generar_cash_flow_statement_growth_quarter_nyse_nasdaq()

C:\Users\mging\AppData\Local\Temp\ipykernel_15488\1800187676.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [36]:
cash_flow_statement_growth_quarter_nyse_nasdaq = pd.read_parquet("archivos/cash_flow_statement_growth_quarter_nyse_nasdaq.parquet")
cash_flow_statement_growth_quarter_nyse_nasdaq

,symbol,date,fiscalYear,period,reportedCurrency,growthNetIncome,growthDepreciationAndAmortization,growthDeferredIncomeTax,growthStockBasedCompensation,growthChangeInWorkingCapital,...,growthOperatingCashFlow,growthCapitalExpenditure,growthFreeCashFlow,growthNetDebtIssuance,growthLongTermNetDebtIssuance,growthShortTermNetDebtIssuance,growthNetStockIssuance,growthPreferredDividendsPaid,growthIncomeTaxesPaid,growthInterestPaid
0,HMY,2025-06-30,2025,Q4,ZAR,-0.169276,-0.005766,0.0,0.049853,0.795707,...,0.223564,-0.466708,0.006321,-1.0,-1.0,0.0,0.0,-1.445561,0.0,-0.074627
1,HMY,2024-12-31,2025,Q2,ZAR,1.946007,0.098146,0.0,0.000000,-3.084416,...,0.176776,-0.060927,0.304000,0.0,0.0,0.0,0.0,0.584551,0.0,0.000000
2,HMY,2024-06-30,2024,Q4,ZAR,-0.549493,-0.071399,0.0,-1.000000,0.720762,...,0.237312,-0.171148,0.319156,1.0,1.0,0.0,0.0,-112.050807,0.0,0.000000
3,HMY,2023-12-31,2024,Q2,ZAR,0.985911,0.430030,1.0,-0.673142,-43.406663,...,0.016272,0.031547,0.082381,0.0,0.0,0.0,0.0,-477.483899,0.0,0.000000
4,HMY,2023-06-30,2023,Q4,ZAR,0.620990,-0.085667,0.0,1.842323,0.000000,...,1.245677,-0.095447,5.972461,0.0,0.0,0.0,0.0,0.993312,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594536,TENKR,2023-06-30,2023,Q2,USD,0.007472,0.000000,0.0,0.000000,5.756335,...,0.406063,0.000000,0.406063,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
594537,TENKR,2023-03-31,2023,Q1,USD,0.606374,0.000000,0.0,0.000000,-1.324979,...,-2.518992,0.000000,-2.518992,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
594538,TENKR,2022-12-31,2022,Q4,USD,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
594539,TENKR,2022-09-30,2022,Q3,USD,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000


In [42]:
def generar_financial_growth_quarter_nyse_nasdaq(
    symbols_parquet: str = "archivos/symbols_nyse_nasdaq.parquet",
    out_path: str = "archivos/financial_growth_quarter_nyse_nasdaq.parquet") -> Path:
    """
    Descarga todos los key metrics trimestrales para los símbolos del parquet
    y guarda un único DataFrame consolidado en parquet.
    """
    # 1) Leer símbolos
    symbols = (
        pd.read_parquet(symbols_parquet)["symbol"]
        .dropna().astype(str).str.upper().unique().tolist()
    )

    # 2) Cliente
    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointStable(api_key)

    # 3) Descarga con manejo de timeout
    frames = []
    for sym in symbols:
        try:
            data = client.get(
                "financial-growth",
                params={"symbol": sym, "period": "quarter", "limit": 1000, "timeout": 30}
            )
            if data:
                df_sym = pd.DataFrame(data)
                df_sym["symbol"] = sym
                frames.append(df_sym)
        except requests.exceptions.ReadTimeout:
            continue
        except Exception as e:
            continue

        # opcional: dormir un poco entre requests para aliviar API
        time.sleep(0.2)

    df_total = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # 4) Guardar con seguridad
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    string_cols_keep = {"symbol", "date", "period", "reportedCurrency"}
    INT64_MAX = 2**63 - 1

    
    for col in df_total.columns:
        if df_total[col].dtype == "object" and col not in string_cols_keep:
            try:
                series = pd.to_numeric(df_total[col], errors="coerce")
                if series.max(skipna=True) > INT64_MAX:
                    df_total[col] = df_total[col].astype(str)
                else:
                    df_total[col] = series
            except Exception:
                df_total[col] = df_total[col].astype(str)

    df_total.to_parquet(out_path, index=False, engine="pyarrow")

    return out_path

In [43]:
generar_financial_growth_quarter_nyse_nasdaq()

In [44]:
financial_growth_quarter_nyse_nasdaq = pd.read_parquet("archivos/financial_growth_quarter_nyse_nasdaq.parquet")
financial_growth_quarter_nyse_nasdaq

,symbol,date,fiscalYear,period,reportedCurrency,revenueGrowth,grossProfitGrowth,ebitgrowth,operatingIncomeGrowth,netIncomeGrowth,...,fiveYShareholdersEquityGrowthPerShare,threeYShareholdersEquityGrowthPerShare,tenYDividendperShareGrowthPerShare,fiveYDividendperShareGrowthPerShare,threeYDividendperShareGrowthPerShare,ebitdaGrowth,growthCapitalExpenditure,tenYBottomLineNetIncomeGrowthPerShare,fiveYBottomLineNetIncomeGrowthPerShare,threeYBottomLineNetIncomeGrowthPerShare
0,HMY,2025-06-30,2025,Q4,ZAR,-0.010393,0.009808,0.093316,0.035089,-0.169276,...,0.748762,0.000000,0.0,0.0,0.000000,0.074317,-0.466708,18.309255,44.440443,0.000000
1,HMY,2024-12-31,2025,Q2,ZAR,0.245715,0.887379,1.243314,0.482365,1.946007,...,0.674774,0.391067,0.0,0.0,2.562750,0.869482,-0.060927,74.645226,71.559756,88.119624
2,HMY,2024-06-30,2024,Q4,ZAR,-0.055411,-0.093097,-0.376435,0.369318,-0.549493,...,0.517988,0.255355,0.0,0.0,3.135524,-0.305333,-0.171148,16.899764,13.221666,71.181403
3,HMY,2023-12-31,2024,Q2,ZAR,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.279068,0.0,0.0,0.000000,0.000000,0.031547,478.007258,0.000000,17.913566
4,HMY,2023-06-30,2023,Q4,ZAR,0.000000,0.000000,0.000000,0.000000,0.000000,...,-1.000000,-1.000000,-1.0,0.0,0.000000,0.000000,-0.095447,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
652360,TENKR,2021-03-31,2021,Q1,USD,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
652361,AFTR-WT,2023-03-31,2023,Q1,USD,0.000000,0.000000,0.000000,0.000000,-0.892565,...,-2.494625,-2.494625,0.0,0.0,0.000000,0.000000,0.000000,-0.642468,-0.642468,-0.642468
652362,AFTR-WT,2022-09-30,2022,Q3,USD,0.000000,0.000000,0.000000,0.000000,-0.168025,...,0.516086,0.516086,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
652363,AFTR-WT,2022-03-31,2022,Q1,USD,0.000000,0.000000,0.000000,0.000000,0.201959,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
